# Phase 3 — TRL SFT on the SPY OpenEnv (Hackathon)

**Runtime:** enable a **GPU** (T4 or better).

**Secrets (Colab):** add `HF_TOKEN` (read Hub) and optionally `HUGGINGFACEHUB_API_TOKEN` if you push adapters.

**Flow:** clone repo → install TRL stack → collect trajectories from your **live Space** → run `SFTTrainer` → view loss plot.

**Open in Colab:** use the notebook URL on GitHub (`colab/phase3_trl_sft.ipynb`).

In [ ]:
import os
import subprocess

os.environ["PYTHONUTF8"] = "1"

REPO_URL = "https://github.com/kunaljaiswal2461-lab/metaOpenNV_V2.git"
DIR = "metaOpenNV_V2"
if not os.path.isdir(DIR):
    subprocess.check_call(["git", "clone", REPO_URL])
os.chdir(DIR)

In [ ]:
!pip install -q -r requirements-trl.txt

In [ ]:
import os
from google.colab import userdata

# Live Space that exposes /reset and /step
os.environ["SPACE_URL"] = "https://huggingface.co/spaces/Kj2461/metaOpenNV_V2"

try:
    tok = userdata.get("HF_TOKEN")
    os.environ["HF_TOKEN"] = tok
except Exception:
    print("Set HF_TOKEN in Colab secrets if you need authenticated Hub access.")

In [ ]:
# Roll the environment over HTTP; teacher = SMA20 heuristic (see trl_data/prompt_utils.py)
!python scripts/collect_sft_dataset.py --episodes 12 --max-steps 400 --task-name risk_aware_trading --out data/trl_sft_train.jsonl

In [ ]:
# SFT a small instruct model (adjust --model-id if VRAM is tight)
!python scripts/trl_sft_train.py --data data/trl_sft_train.jsonl --model-id Qwen/Qwen2.5-0.5B-Instruct --epochs 1 --output-dir results/phase3_lora

In [ ]:
from IPython.display import Image, display
display(Image("results/trl_sft_loss.png"))

In [ ]:
# Optional: compare base vs fine-tuned on the live env (writes results/phase3_eval_*).
# Runs ~5 episodes per model; bump to 10-20 once you trust the wiring.
!python scripts/eval_llm_on_env.py \
    --models base=Qwen/Qwen2.5-0.5B-Instruct sft=results/phase3_lora \
    --episodes 5 --max-steps 200 --task-name risk_aware_trading

from IPython.display import Image, display
display(Image("results/phase3_eval_bar.png"))